# Week 04 · City Trips: от таблицы к визуальному выводу
**BrainCode Academy · Machine learning and Artificial Intelligence**  
**Преподаватель: Assylbek Gizatov**

Продолжение Titanic (Week 02) и City Trips (Week 03). Повторяем `loc`, `groupby`, `describe`, пропуски и доли; добавляем осознанный выбор графика, сравнение групп и корреляцию. SQL и обучение моделей здесь не нужны.

**Два занятия:** A — повторение и распределения (разделы 1–5); B — сравнения и связи (6–10). Основной маршрут: 1–8. ECDF и Plotly — бонус, не обязательный барьер.

**Данные:** та же фиксированная синтетическая выгрузка City Trips, 5–11 октября 2026. Это учебная конструкция, не наблюдения реального города. Отмены не имеют стоимости/длительности; справочник намеренно неполный. Данные встроены: интернет для загрузки не требуется.

Запускайте сверху вниз. Нужны pandas, numpy, matplotlib, seaborn (Plotly только для бонуса). При отсутствии библиотек установите их в окружение Jupyter: `%pip install pandas numpy matplotlib seaborn plotly`. Выходные файлы сохраняются в `week04_outputs`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from io import StringIO
from pathlib import Path
import json
from IPython.display import display
sns.set_theme(style="whitegrid", font="DejaVu Sans")
pd.set_option("display.max_columns", 12)
OUTPUT = Path("week04_outputs")
OUTPUT.mkdir(exist_ok=True)

## 1. Знакомые данные и короткая проверка

In [ ]:
TRIPS_CSV = 'trip_id,requested_at,zone_id,channel,status,distance_km,duration_min,fare_kzt\n1001,2026-10-05 06:00,Z01,app,cancelled,,,\n1002,2026-10-05 06:30,Z03,phone,completed,3.0,12.7,1100\n1003,2026-10-05 07:00,Z02,app,completed,15.8,56.2,3450\n1004,2026-10-05 07:30,Z03,app,completed,11.8,32.6,2650\n1005,2026-10-05 08:00,Z01,phone,completed,9.7,28.7,2250\n1006,2026-10-05 08:30,Z05,web,completed,15.1,59.4,3150\n1007,2026-10-05 09:00,Z04,,completed,10.5,28.7,2450\n1008,2026-10-05 09:30,Z04,app,completed,1.6,9.2,800\n1009,2026-10-05 10:00,Z01,phone,cancelled,,,\n1010,2026-10-05 10:30,Z03,phone,completed,8.6,,2050\n1011,2026-10-05 11:00,Z02,app,completed,15.8,53.8,3400\n1012,2026-10-05 11:30,Z03,app,completed,5.1,13.5,1500\n1013,2026-10-05 12:00,Z03,app,completed,13.6,39.4,2900\n1014,2026-10-05 12:30,Z04,app,completed,13.3,47.2,2950\n1015,2026-10-05 13:00,Z01,app,completed,18.5,59.1,3900\n1016,2026-10-05 13:30,Z02,phone,completed,8.7,25.1,1950\n1017,2026-10-05 14:00,Z01,phone,cancelled,,,\n1018,2026-10-05 14:30,Z05,phone,cancelled,,,\n1019,2026-10-05 15:00,Z05,app,completed,4.7,19.4,1450\n1020,2026-10-05 15:30,Z04,phone,completed,10.9,44.6,2450\n1021,2026-10-05 16:00,Z02,phone,completed,8.8,35.5,2050\n1022,2026-10-05 16:30,Z02,web,completed,17.9,64.8,3650\n1023,2026-10-05 17:00,Z04,web,completed,7.6,24.7,1800\n1024,2026-10-05 17:30,Z02,app,completed,1.2,11.1,650\n1025,2026-10-06 06:00,Z03,app,cancelled,,,\n1026,2026-10-06 06:30,Z01,app,completed,10.7,32.5,2400\n1027,2026-10-06 07:00,Z01,app,completed,11.3,44.2,2550\n1028,2026-10-06 07:30,Z04,web,completed,10.6,32.0,2450\n1029,2026-10-06 08:00,Z04,app,completed,2.7,15.9,1100\n1030,2026-10-06 08:30,Z04,app,completed,10.0,29.1,2250\n1031,2026-10-06 09:00,Z02,web,completed,7.9,23.7,2000\n1032,2026-10-06 09:30,Z02,phone,completed,1.6,8.5,750\n1033,2026-10-06 10:00,Z02,web,cancelled,,,\n1034,2026-10-06 10:30,Z03,app,completed,11.9,32.6,2600\n1035,2026-10-06 11:00,Z02,app,cancelled,,,\n1036,2026-10-06 11:30,Z03,app,completed,2.4,9.5,900\n1037,2026-10-06 12:00,Z01,app,completed,15.7,49.3,3250\n1038,2026-10-06 12:30,Z03,app,completed,1.6,8.2,800\n1039,2026-10-06 13:00,Z05,app,completed,12.6,32.6,2800\n1040,2026-10-06 13:30,Z02,app,completed,6.5,20.2,1700\n1041,2026-10-06 14:00,Z05,phone,cancelled,,,\n1042,2026-10-06 14:30,Z01,app,completed,10.2,32.8,2350\n1043,2026-10-06 15:00,Z04,phone,completed,15.9,45.4,3400\n1044,2026-10-06 15:30,Z04,app,completed,14.8,55.7,3050\n1045,2026-10-06 16:00,Z02,app,completed,10.9,37.0,2400\n1046,2026-10-06 16:30,Z03,app,completed,14.2,50.4,3100\n1047,2026-10-06 17:00,Z01,app,completed,3.0,15.0,1000\n1048,2026-10-06 17:30,Z01,app,completed,7.4,30.2,1750\n1049,2026-10-07 06:00,Z03,app,cancelled,,,\n1050,2026-10-07 06:30,Z04,app,completed,6.6,18.3,1700\n1051,2026-10-07 07:00,Z04,web,completed,5.6,22.2,1450\n1052,2026-10-07 07:30,Z01,app,cancelled,,,\n1053,2026-10-07 08:00,Z04,app,completed,17.4,51.2,3700\n1054,2026-10-07 08:30,Z03,phone,completed,1.9,10.7,900\n1055,2026-10-07 09:00,Z02,,completed,16.4,58.0,3550\n1056,2026-10-07 09:30,Z05,app,completed,6.2,22.8,1700\n1057,2026-10-07 10:00,Z02,app,cancelled,,,\n1058,2026-10-07 10:30,Z04,app,completed,17.8,61.1,3800\n1059,2026-10-07 11:00,Z02,app,completed,1.6,8.8,700\n1060,2026-10-07 11:30,Z04,app,completed,11.9,28.8,2650\n1061,2026-10-07 12:00,Z03,web,completed,6.4,19.9,1650\n1062,2026-10-07 12:30,Z02,app,completed,8.5,22.7,2050\n1063,2026-10-07 13:00,Z01,phone,completed,10.6,37.9,2350\n1064,2026-10-07 13:30,Z01,app,completed,10.6,36.0,2350\n1065,2026-10-07 14:00,Z05,app,cancelled,,,\n1066,2026-10-07 14:30,Z04,app,completed,12.4,49.0,2700\n1067,2026-10-07 15:00,Z02,app,completed,14.1,36.5,3100\n1068,2026-10-07 15:30,Z05,app,completed,18.0,62.7,3850\n1069,2026-10-07 16:00,Z05,app,cancelled,,,\n1070,2026-10-07 16:30,Z05,web,completed,5.2,21.4,1400\n1071,2026-10-07 17:00,Z02,app,completed,9.1,37.4,2200\n1072,2026-10-07 17:30,Z03,app,completed,12.1,34.5,2650\n1073,2026-10-08 06:00,Z04,phone,cancelled,,,\n1074,2026-10-08 06:30,Z04,phone,completed,12.3,30.5,2800\n1075,2026-10-08 07:00,Z01,phone,completed,2.9,15.2,1100\n1076,2026-10-08 07:30,Z03,app,completed,17.7,54.0,3700\n1077,2026-10-08 08:00,Z04,app,completed,17.9,,3700\n1078,2026-10-08 08:30,Z02,app,completed,1.9,12.3,800\n1079,2026-10-08 09:00,Z05,app,completed,6.1,17.8,1700\n1080,2026-10-08 09:30,Z03,phone,completed,12.4,47.6,2850\n1081,2026-10-08 10:00,Z05,web,cancelled,,,\n1082,2026-10-08 10:30,Z05,app,completed,10.1,28.3,2250\n1083,2026-10-08 11:00,Z01,app,completed,12.8,46.6,2800\n1084,2026-10-08 11:30,Z01,app,completed,10.4,34.3,2300\n1085,2026-10-08 12:00,Z05,app,completed,16.3,55.8,3500\n1086,2026-10-08 12:30,Z02,app,cancelled,,,\n1087,2026-10-08 13:00,Z02,app,completed,10.8,30.0,2450\n1088,2026-10-08 13:30,Z05,app,completed,2.4,15.4,1050\n1089,2026-10-08 14:00,Z02,phone,cancelled,,,\n1090,2026-10-08 14:30,Z01,app,completed,10.0,32.8,2400\n1091,2026-10-08 15:00,Z04,app,completed,6.6,20.1,1750\n1092,2026-10-08 15:30,Z05,phone,completed,7.3,29.9,1750\n1093,2026-10-08 16:00,Z99,phone,completed,12.6,31.4,2700\n1094,2026-10-08 16:30,Z02,phone,completed,8.9,25.6,2050\n1095,2026-10-08 17:00,Z02,app,completed,14.6,40.9,3200\n1096,2026-10-08 17:30,Z02,app,completed,13.6,45.9,3000\n1097,2026-10-09 06:00,Z04,app,cancelled,,,\n1098,2026-10-09 06:30,Z04,app,completed,15.7,55.7,3350\n1099,2026-10-09 07:00,Z02,app,completed,17.2,56.4,3650\n1100,2026-10-09 07:30,Z04,app,completed,11.2,40.4,2550\n1101,2026-10-09 08:00,Z04,app,completed,1.4,10.3,800\n1102,2026-10-09 08:30,Z04,app,completed,1.5,8.2,750\n1103,2026-10-09 09:00,Z03,app,cancelled,,,\n1104,2026-10-09 09:30,Z03,,completed,14.5,54.7,3050\n1105,2026-10-09 10:00,Z05,app,cancelled,,,\n1106,2026-10-09 10:30,Z04,phone,completed,2.7,12.3,1050\n1107,2026-10-09 11:00,Z03,app,completed,5.9,18.9,1650\n1108,2026-10-09 11:30,Z03,app,completed,17.5,54.2,3650\n1109,2026-10-09 12:00,Z02,app,completed,5.6,26.1,1400\n1110,2026-10-09 12:30,Z04,app,completed,9.5,31.6,2250\n1111,2026-10-09 13:00,Z01,app,completed,15.8,45.0,3250\n1112,2026-10-09 13:30,Z04,phone,completed,13.8,46.9,3000\n1113,2026-10-09 14:00,Z04,app,cancelled,,,\n1114,2026-10-09 14:30,Z03,app,completed,10.0,32.3,2300\n1115,2026-10-09 15:00,Z04,app,completed,2.6,14.3,1050\n1116,2026-10-09 15:30,Z01,web,completed,16.3,50.5,3350\n1117,2026-10-09 16:00,Z05,web,completed,1.7,8.6,850\n1118,2026-10-09 16:30,Z01,app,completed,16.1,58.5,3350\n1119,2026-10-09 17:00,Z03,phone,completed,6.8,28.8,1800\n1120,2026-10-09 17:30,Z04,app,cancelled,,,\n1121,2026-10-10 06:00,Z03,phone,cancelled,,,\n1122,2026-10-10 06:30,Z01,app,completed,15.0,54.6,3250\n1123,2026-10-10 07:00,Z04,app,completed,6.4,24.0,1700\n1124,2026-10-10 07:30,Z04,app,completed,4.1,19.3,1300\n1125,2026-10-10 08:00,Z01,app,completed,9.7,30.0,2200\n1126,2026-10-10 08:30,Z03,phone,completed,2.8,13.1,1000\n1127,2026-10-10 09:00,Z01,app,completed,9.3,23.6,2250\n1128,2026-10-10 09:30,Z04,phone,completed,6.9,19.1,1650\n1129,2026-10-10 10:00,Z02,app,cancelled,,,\n1130,2026-10-10 10:30,Z04,app,completed,5.1,21.4,1350\n1131,2026-10-10 11:00,Z03,phone,completed,13.7,41.6,2900\n1132,2026-10-10 11:30,Z05,app,completed,8.8,36.6,2000\n1133,2026-10-10 12:00,Z04,app,completed,4.6,18.9,1400\n1134,2026-10-10 12:30,Z03,app,completed,4.2,15.8,1250\n1135,2026-10-10 13:00,Z01,web,completed,12.4,49.7,2700\n1136,2026-10-10 13:30,Z04,app,completed,9.4,29.4,2200\n1137,2026-10-10 14:00,Z03,app,cancelled,,,\n1138,2026-10-10 14:30,Z04,app,completed,18.0,54.8,3800\n1139,2026-10-10 15:00,Z01,app,completed,10.9,42.8,2400\n1140,2026-10-10 15:30,Z02,app,completed,15.5,49.2,3200\n1141,2026-10-10 16:00,Z04,app,completed,2.6,10.5,1000\n1142,2026-10-10 16:30,Z03,app,completed,12.4,40.9,2750\n1143,2026-10-10 17:00,Z03,app,completed,6.7,18.0,1650\n1144,2026-10-10 17:30,Z03,app,completed,18.0,60.0,3800\n1145,2026-10-11 06:00,Z03,phone,cancelled,,,\n1146,2026-10-11 06:30,Z04,app,completed,3.9,17.2,1300\n1147,2026-10-11 07:00,Z05,app,completed,8.1,30.0,1950\n1148,2026-10-11 07:30,Z01,app,completed,4.4,12.8,1300\n1149,2026-10-11 08:00,Z03,app,completed,1.5,9.6,700\n1150,2026-10-11 08:30,Z03,phone,completed,6.3,25.2,1750\n1151,2026-10-11 09:00,Z02,phone,completed,1.7,10.2,850\n1152,2026-10-11 09:30,Z01,app,completed,5.0,13.6,1400\n1153,2026-10-11 10:00,Z04,phone,cancelled,,,\n1154,2026-10-11 10:30,Z04,app,cancelled,,,\n1155,2026-10-11 11:00,Z05,app,completed,2.1,9.7,850\n1156,2026-10-11 11:30,Z05,app,completed,4.1,17.5,1300\n1157,2026-10-11 12:00,Z04,app,completed,16.8,50.6,3600\n1158,2026-10-11 12:30,Z02,web,completed,12.1,32.5,2650\n1159,2026-10-11 13:00,Z04,web,completed,2.0,10.2,850\n1160,2026-10-11 13:30,Z01,app,completed,12.4,37.7,2800\n1161,2026-10-11 14:00,Z04,app,cancelled,,,\n1162,2026-10-11 14:30,Z03,app,completed,5.3,22.8,1400\n1163,2026-10-11 15:00,Z03,app,completed,5.4,15.2,1400\n1164,2026-10-11 15:30,Z02,app,completed,4.2,12.7,1350\n1165,2026-10-11 16:00,Z02,app,completed,12.4,30.7,2700\n1166,2026-10-11 16:30,Z01,phone,completed,18.2,58.0,3750\n1167,2026-10-11 17:00,Z04,app,completed,13.2,44.8,2900\n1168,2026-10-11 17:30,Z03,app,completed,15.3,39.0,3300\n1009,2026-10-05 10:00,Z01,phone,cancelled,,,\n1044,2026-10-06 15:30,Z04,app,completed,14.8,55.7,3050\n'
ZONES_JSON = '[\n  {\n    "zone_id": "Z01",\n    "zone_name": "Центр",\n    "zone_type": "Смешанная"\n  },\n  {\n    "zone_id": "Z02",\n    "zone_name": "Университет",\n    "zone_type": "Учебная"\n  },\n  {\n    "zone_id": "Z03",\n    "zone_name": "Вокзал",\n    "zone_type": "Транспортная"\n  },\n  {\n    "zone_id": "Z04",\n    "zone_name": "Жилой район",\n    "zone_type": "Жилая"\n  },\n  {\n    "zone_id": "Z05",\n    "zone_name": "Деловой район",\n    "zone_type": "Деловая"\n  }\n]'
raw = pd.read_csv(StringIO(TRIPS_CSV))
zones = pd.DataFrame(json.loads(ZONES_JSON))
print("Исходные строки:", len(raw))
display(raw.head(3))

In [ ]:
trips = raw.drop_duplicates().copy()
trips["requested_at"] = pd.to_datetime(trips["requested_at"], errors="raise")
assert trips["trip_id"].is_unique, "Повторяющийся ID требует проверки"
joined = trips.merge(zones, on="zone_id", how="left", validate="many_to_one")
joined["zone_report"] = joined["zone_name"].fillna("Неизвестная зона")
joined["channel_report"] = joined["channel"].fillna("unknown")
completed = joined.loc[joined["status"].eq("completed")].copy()
assert len(joined) == len(trips)
print("Уникальных заявок:", len(trips), "| Завершено:", len(completed))
display(completed[["distance_km", "duration_min", "fare_kzt"]].describe().round(2))

In [ ]:
quality = completed[["distance_km", "duration_min", "fare_kzt"]].isna().sum()
display(quality.rename("missing_completed"))
durations = completed["duration_min"].dropna()
print("Длительность известна для", len(durations), "из", len(completed), "поездок")

## 2. Сначала вопрос, потом график
Сколько заявок по каналам → столбцы. Как распределена длительность → гистограмма. Чем отличаются зоны → boxplot + размеры групп. Как связаны две величины → scatter.

In [ ]:
counts = joined["channel_report"].value_counts()
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(counts.index, counts.values, color="#2563EB")
ax.set(title="Заявки по каналам · учебные данные", xlabel="Канал", ylabel="Заявок", ylim=(0, counts.max()*1.2))
for i, value in enumerate(counts):
    ax.text(i, value + 1, str(value), ha="center")
fig.tight_layout()
plt.show()

## 3. Гистограмма: форма распределения
Измените `bins` с 8 на 16. Что изменилось в рисунке, а что осталось тем же в исходных данных?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(durations, bins=10, color="#0F766E", edgecolor="white")
ax.axvline(durations.median(), color="#B91C1C", linestyle="--", label=f"Медиана {durations.median():.1f} мин")
ax.set(title=f"Длительность завершённых поездок (n={len(durations)})", xlabel="Минуты", ylabel="Поездок в интервале")
ax.legend()
fig.tight_layout()
plt.show()

## 4. Бонус: ECDF отвечает «какая доля не дольше X?»

In [ ]:
threshold = 30
share = durations.le(threshold).mean()
fig, ax = plt.subplots(figsize=(8, 4))
sns.ecdfplot(x=durations, ax=ax, color="#2563EB")
ax.axvline(threshold, color="#B91C1C", linestyle="--")
ax.axhline(share, color="#B91C1C", linestyle=":")
ax.set(xlabel="Длительность, минуты", ylabel="Доля поездок с длительностью ≤ X", title="ECDF · только известные длительности")
fig.tight_layout()
plt.show()
print(f"Не дольше {threshold} мин: {durations.le(threshold).sum()}/{len(durations)} = {share:.1%}")

## 5. Мини-практика A
1. Постройте гистограмму `fare_kzt`, подпишите тенге и медиану.
2. Выведите число известных оплат.
3. Сравните варианты 6 и 15 интервалов.
4. Напишите два наблюдения с числами и одно ограничение. Используйте ИИ для объяснения ошибки, но самостоятельно объясните каждую строку.

In [ ]:
# Рабочая ячейка: используйте completed["fare_kzt"].dropna(), ax.hist и ax.axvline.
# Создайте собственные fig, ax, чтобы не дорисовывать старый график.

## 6. Сравнение зон: медиана не рассказывает всё

In [ ]:
by_zone = completed.groupby("zone_report").agg(n=("duration_min", "count"), median_min=("duration_min", "median"), mean_min=("duration_min", "mean"))
order = by_zone.sort_values("median_min").index.tolist()
display(by_zone.loc[order].round(1))
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=completed, x="duration_min", y="zone_report", order=order, color="#93C5FD", ax=ax)
ax.set(title="Разброс длительности по зонам · учебные данные", xlabel="Минуты", ylabel="Зона")
fig.tight_layout()
plt.show()

## 7. Scatter и корреляция: связь, а не причина

In [ ]:
pairs = completed.dropna(subset=["distance_km", "duration_min"])
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=pairs, x="distance_km", y="duration_min", hue="zone_report", alpha=0.7, s=45, ax=ax)
ax.set(title=f"Дистанция и длительность (пар n={len(pairs)})", xlabel="Дистанция, км", ylabel="Длительность, мин")
ax.legend(title="Зона", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
plt.show()

In [ ]:
pearson = pairs["distance_km"].corr(pairs["duration_min"], method="pearson")
spearman = pairs["distance_km"].rank().corr(pairs["duration_min"].rank())
print(f"n={len(pairs)}, Pearson={pearson:.3f}, Spearman={spearman:.3f}")
cols = ["distance_km", "duration_min", "fare_kzt"]
corr_data = completed[cols].dropna()
fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(corr_data.corr(), annot=True, fmt=".2f", vmin=-1, vmax=1, center=0, cmap="coolwarm", ax=ax)
ax.set_title(f"Pearson · общие полные строки, n={len(corr_data)}")
fig.tight_layout()
plt.show()

## 8. Честная доля: все заявки в знаменателе

In [ ]:
report_data = joined.assign(cancelled=joined["status"].eq("cancelled"))
report = report_data.groupby("zone_report").agg(requests=("trip_id", "size"), cancelled=("cancelled", "sum"))
report["cancel_pct"] = report["cancelled"] / report["requests"] * 100
report = report.sort_values("cancel_pct")
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(report.index, report["cancel_pct"], color="#0F766E")
ax.set(xlabel="Доля отмен, %", ylabel="Зона", title="Отмены: числитель / все заявки зоны", xlim=(0, 100))
for i, row in enumerate(report.itertuples()):
    ax.text(row.cancel_pct + 1, i, f"{row.cancelled}/{row.requests}", va="center")
fig.tight_layout()
fig.savefig(OUTPUT / "cancellations.png", dpi=160, bbox_inches="tight")
plt.show()
display(report.round(1))

## 9. Временной график и интерактивный бонус

In [ ]:
daily = joined.groupby(joined["requested_at"].dt.date).size().sort_index()
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(daily.index, daily.values, marker="o", color="#2563EB")
ax.set(title="Заявки по дням · синтетический график генерации", xlabel="Дата", ylabel="Заявок", ylim=(0, daily.max()*1.2))
fig.autofmt_xdate()
fig.tight_layout()
plt.show()
print("Одинаковые значения здесь отражают учебную конструкцию, не стабильность реального спроса.")

In [ ]:
try:
    import plotly.express as px
except ImportError:
    print("Бонус пропущен: установите plotly, если нужен HTML с наведением.")
else:
    interactive = px.scatter(pairs, x="distance_km", y="duration_min", color="zone_report", hover_data=["trip_id", "fare_kzt"], title="City Trips · учебные данные")
    interactive.update_layout(xaxis_title="Дистанция, км", yaxis_title="Длительность, мин")
    interactive.write_html(OUTPUT / "city_trips_interactive.html", include_plotlyjs=True)
    print("HTML сохранён: откройте файл в браузере, интернет не нужен.")

## 10. Самостоятельный мини-отчёт
1. Сформулируйте вопрос о цене/дистанции, ответьте таблицей и подходящим графиком.
2. Сравните медиану оплаты по каналам: покажите размер каждой группы.
3. Рассчитайте долю завершённых поездок **среди всех заявок каждого канала**. Не подменяйте её долей канала среди завершённых.
4. Измените один параметр рисунка (bins, alpha или figsize), объясните эффект.
5. Сохраните PNG и CSV. Сформулируйте: наблюдение с числом → ограничение → следующий вопрос.
6. ИИ можно и нужно использовать для объяснения. Запишите запрос и одну проверку ответа ИИ на данных. Не сдавайте код, который не можете объяснить.

**Защита 3–5 минут:** вопрос, выбор графика, знаменатель, вывод, ограничение. Бонус: сравните связь дистанции/длительности внутри зон; маленькие группы не интерпретируйте уверенно.

**Самопроверка:** единицы на осях; известное n; отмены не стали нулями; нет причинных заявлений; результаты названы синтетическими.

In [ ]:
# Ваш анализ здесь: используйте groupby, agg, count/size, median и plt.subplots.
# Отдельную исходную таблицу joined не меняйте: создавайте новые результаты.

In [ ]:
assert report["requests"].sum() == len(joined), "Потеряны заявки при группировке"
assert report["cancelled"].sum() == joined["status"].eq("cancelled").sum()
report.to_csv(OUTPUT / "zone_report.csv", encoding="utf-8-sig")
print("Готово. Результаты:", OUTPUT.resolve())

### Справка
[Matplotlib](https://matplotlib.org/stable/users/explain/quick_start.html) · [Seaborn](https://seaborn.pydata.org/tutorial.html) · [pandas groupby](https://pandas.pydata.org/docs/user_guide/groupby.html)

Новый материал: форма распределения, осмысленный выбор рисунка, сравнение групп, ограничения корреляции. Основа прежняя: DataFrame, фильтр, группировка, агрегат.